# 04 · IDUN raw → LSL alignment

Generate `idun_repair_root/idun_repair_summary.csv`, then align native IDUN EEG with cap EEG in matching XDFs under `processing_data_root`. No earlier synchronization step is required.


In [1]:
import importlib
import sys
from pathlib import Path

from IPython.display import display

for PROJECT_ROOT in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (PROJECT_ROOT / 'helpers').is_dir():
        sys.path.insert(0, str(PROJECT_ROOT))
        break
else:
    raise FileNotFoundError('Could not locate the ImBrAid/helpers folder')

from helpers.idun import raw_lsl_repair as repair
from helpers.project_config import load_project_config
from helpers.sync_quality import idun_lsl_alignment as alignment
importlib.reload(repair)
importlib.reload(alignment)

CONFIG = load_project_config(PROJECT_ROOT / 'config.json')
DATA_ROOT = CONFIG.path('paths', 'processing_data_root')
REPAIR_ROOT = CONFIG.path('paths', 'idun_repair_root')
ALIGNMENT_CSV = CONFIG.path('files', 'idun_lsl_alignment_csv')
OVERWRITE = True
PARTICIPANTS = None
VISITS = None
CHUNK_SAMPLES = 20
REPAIR_SEARCH_RADIUS_S = 30.0
CAP_STREAM_NAME = CONFIG.value('modalities', 'cap_eeg_stream_name')
BASE_HZ = 50.0
FEATURE_HZ = 10.0
WINDOW_S = 300.0
STEP_S = 150.0
COARSE_SEARCH_RADIUS_S = 30.0
LOCAL_SEARCH_RADIUS_S = 2.0
MINIMUM_PEAK_CORRELATION = 0.35
MINIMUM_PEAK_MARGIN = 0.02
REVIEW_DRIFT_PPM = 2000.0
MAXIMUM_DRIFT_PPM = 10000.0

In [2]:
repair_jobs = repair.discover_idun_jobs(
    DATA_ROOT, project=CONFIG.project, participants=PARTICIPANTS, visits=VISITS,
)

display(repair_jobs[['pid', 'visit', 'discovery_status']])


,pid,visit,discovery_status
0,A7G5SM,VX,missing_raw
1,ABVLGQ,VX,ready
2,B9NYQ8,VX,missing_raw
3,CRVWF9,VX,ready
4,CRYSEP,VX,ready
5,CU6DM4,VX,ready
6,DUJL2L,VX,ready
7,E4MKXT,VX,ready
8,EASFXM,VX,ready
9,FEDZAG,VX,ready


In [3]:
repair_summary = repair.run_idun_repairs(
    repair_jobs, REPAIR_ROOT, chunk_samples=CHUNK_SAMPLES,
    search_radius_s=REPAIR_SEARCH_RADIUS_S,
)

display(repair_summary[[
    'pid', 'visit', 'repair_status', 'usable_relative_time',
    'quality_grade', 'review_required', 'matched_chunk_ratio', 'warnings',
]])


,pid,visit,repair_status,usable_relative_time,quality_grade,review_required,matched_chunk_ratio,warnings
0,A7G5SM,VX,unavailable,False,D,True,NaN,Discovery status: missing_raw
1,ABVLGQ,VX,repaired,True,A,False,1.0,18 chunk transitions were not in raw acquisiti...
2,B9NYQ8,VX,unavailable,False,D,True,NaN,Discovery status: missing_raw
3,CRVWF9,VX,repaired,True,A,False,1.0,272 chunk transitions were not in raw acquisit...
4,CRYSEP,VX,repaired,True,A,False,1.0,120 chunk transitions were not in raw acquisit...
5,CU6DM4,VX,repaired,True,A,False,1.0,364 chunk transitions were not in raw acquisit...
6,DUJL2L,VX,repaired,True,A,False,1.0,193 chunk transitions were not in raw acquisit...
7,E4MKXT,VX,repaired,True,A,False,1.0,424 chunk transitions were not in raw acquisit...
8,EASFXM,VX,repaired,True,A,False,1.0,295 chunk transitions were not in raw acquisit...
9,FEDZAG,VX,repaired,True,A,False,1.0,295 chunk transitions were not in raw acquisit...


In [4]:
jobs = alignment.discover_cap_eeg_sync_jobs(
    REPAIR_ROOT, participants=PARTICIPANTS, visits=VISITS,
)
display(jobs[['pid', 'visit', 'discovery_status']])

,pid,visit,discovery_status
0,A7G5SM,VX,idun_repair_unusable
1,ABVLGQ,VX,ready
2,B9NYQ8,VX,idun_repair_unusable
3,CRVWF9,VX,ready
4,CRYSEP,VX,ready
5,CU6DM4,VX,ready
6,DUJL2L,VX,ready
7,E4MKXT,VX,ready
8,EASFXM,VX,ready
9,FEDZAG,VX,ready


In [5]:
mappings = alignment.build_idun_cap_alignments(
    jobs, cap_stream_name=CAP_STREAM_NAME, base_hz=BASE_HZ,
    feature_hz=FEATURE_HZ, window_s=WINDOW_S, step_s=STEP_S,
    coarse_search_radius_s=COARSE_SEARCH_RADIUS_S,
    local_search_radius_s=LOCAL_SEARCH_RADIUS_S,
    minimum_peak_correlation=MINIMUM_PEAK_CORRELATION,
    minimum_peak_margin=MINIMUM_PEAK_MARGIN,
    review_drift_ppm=REVIEW_DRIFT_PPM,
    maximum_drift_ppm=MAXIMUM_DRIFT_PPM,
)
if OVERWRITE:
    alignment.save_idun_cap_alignment_results(
        mappings, alignment_csv=ALIGNMENT_CSV,
    )
display(mappings[[
    'pid', 'visit', 'mapping_status', 'usable', 'quality_grade',
    'review_required', 'selected_cap_feature', 'drift_ppm',
    'n_good_windows', 'fit_residual_p95_s',
    'mapping_reason', 'warnings',
]])

,pid,visit,mapping_status,usable,quality_grade,review_required,selected_cap_feature,drift_ppm,n_good_windows,fit_residual_p95_s,mapping_reason,warnings
0,A7G5SM,VX,unavailable,False,D,True,NaN,NaN,NaN,NaN,Discovery status: idun_repair_unusable,Discovery status: idun_repair_unusable
1,ABVLGQ,VX,affine,True,A,False,T7-T8,-569.032235,17.0,0.018729,Supported affine mapping from IDUN and cap EEG...,
2,B9NYQ8,VX,unavailable,False,D,True,NaN,NaN,NaN,NaN,Discovery status: idun_repair_unusable,Discovery status: idun_repair_unusable
3,CRVWF9,VX,affine,True,A,False,T7-T8,-1107.618332,16.0,0.056666,Supported affine mapping from IDUN and cap EEG...,
4,CRYSEP,VX,affine,True,A,False,T7-T8,-345.416163,6.0,0.028706,Supported affine mapping from IDUN and cap EEG...,
5,CU6DM4,VX,affine,True,A,False,T7-T8,-428.468097,17.0,0.035443,Supported affine mapping from IDUN and cap EEG...,
6,DUJL2L,VX,affine_review,True,B,True,T8-M1,-762.212185,17.0,0.071696,Usable affine mapping with review-level qualit...,median peak correlation was 0.593
7,E4MKXT,VX,affine,True,A,False,T7-T8,-836.290677,16.0,0.047424,Supported affine mapping from IDUN and cap EEG...,
8,EASFXM,VX,affine,True,A,False,T7-T8,-1006.836460,17.0,0.054746,Supported affine mapping from IDUN and cap EEG...,
9,FEDZAG,VX,affine,True,A,False,T7-T8,-857.608382,16.0,0.052702,Supported affine mapping from IDUN and cap EEG...,
